# MMAF-CD — Stage P1b v2: pancreas-centred CT crops (fixed field of view, common orientation) (MSD Task07 + NIH Pancreas-CT)

**Why:** in E2 the pancreatic specialist reached only AUC 0.64 on MSD Task07. Each input was a whole abdominal slice shrunk to 224 px, so a 2–3 cm tumour covers a few pixels. Clinical pipelines first locate the pancreas (for example with TotalSegmentator) and then classify that region. This notebook builds that region-of-interest input.

**What it does (≈ 25–45 min, CPU is enough):**
1. streams `Task07_Pancreas.tar` from Hugging Face (fallback: MONAI S3 mirror); one patient at a time, nothing large is stored
2. MSD: up to **6 tumour slices** (tumour ≥ 50 px) and **3 tumour-free pancreas slices** per case, **cropped to a FIXED square (32 % of the slice width) centred on the pancreas**, every slice rotated so the spine is at the bottom (v2: same field of view and orientation for MSD and NIH), CT window [−100, 240] HU
3. NIH Pancreas-CT (healthy, attach `tahsin/pancreasct-dataset`): the 3 slices with the most pancreas per case, cropped the same way → healthy crops from a second hospital
4. writes `msd_pancreas_crops/{tumor,normal}/*.png` + `metadata.csv` (path, case, label, source, z, crop box) + a sample grid

**Settings:** Accelerator **None (CPU)** · **Internet ON** · input: `tahsin/pancreasct-dataset` (optional but recommended).

**After it finishes:** attach this notebook's output to **E2P** (Add Input → Your Work → Notebooks → this notebook).


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# P1b — pancreas-centred crops from MSD Task07 (tumour + normal) and NIH (normal)
# ════════════════════════════════════════════════════════════════════════════════
import os, sys, time, tarfile, shutil, subprocess, glob
from pathlib import Path
import numpy as np, pandas as pd
try:
    import nibabel as nib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=False); import nibabel as nib
import requests
from PIL import Image

URLS = ["https://huggingface.co/datasets/Novel-BioMedAI/Medical_Segmentation_Decathlon/resolve/main/Task07_Pancreas.tar",
        "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar"]
OUT = Path("/kaggle/working/msd_pancreas_crops")
for d in ("tumor", "normal"): (OUT / d).mkdir(parents=True, exist_ok=True)
TMP = Path("/tmp/msd_pending"); TMP.mkdir(parents=True, exist_ok=True)
N_TUMOR, N_NORMAL, N_NIH, MIN_TUMOR_PX, MIN_PANC_PX = 6, 3, 3, 50, 200
HU_LO, HU_HI = -100.0, 240.0
CROP_FRAC = 0.32                        # v2: FIXED square = 32 % of the slice width (≈ 164 px on 512², ≈ 11–15 cm), same for every source
T0 = time.time(); ROWS = []

def spread(idx, k):
    idx = sorted(int(i) for i in idx)
    if len(idx) <= k: return idx
    return [idx[int(round(j))] for j in np.linspace(0, len(idx) - 1, k)]

def window(sl):
    return np.clip((sl.astype(np.float32) - HU_LO) / (HU_HI - HU_LO), 0, 1)

def orient(img01, mask):
    """v2: rotate the slice (and mask) so the spine is at the bottom, identically for MSD and NIH.
    Bone = brightest pixels; the rotation that puts their centroid (central band) lowest wins."""
    best, bk = -1, 0
    for k in range(4):
        r = np.rot90(img01, k); H, W = r.shape; band = r[:, W // 3: 2 * W // 3]
        ys = np.where(band >= 0.98)[0]
        score = ys.mean() / H if len(ys) > 50 else -1
        if score > best: best, bk = score, k
    return np.ascontiguousarray(np.rot90(img01, bk)), np.ascontiguousarray(np.rot90(mask, bk)), bk

def crop_box(mask2d):
    ys, xs = np.where(mask2d)
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    H, W = mask2d.shape; side = min(int(round(CROP_FRAC * W)), H, W)
    y0 = int(np.clip(round(cy - side / 2), 0, H - side)); x0 = int(np.clip(round(cx - side / 2), 0, W - side))
    return y0, x0, side

def save_crop(img01, box, path):
    y0, x0, s = box; c = img01[y0:y0 + s, x0:x0 + s]
    Image.fromarray((c * 255).astype(np.uint8)).save(path)

# ---------------- MSD Task07 (streamed) ----------------
def process_msd(case, img_path, lab_path):
    lab = np.asarray(nib.load(str(lab_path)).dataobj)
    tum = (lab == 2).sum((0, 1)); pan = (lab >= 1).sum((0, 1))
    zt = spread(np.where(tum >= MIN_TUMOR_PX)[0], N_TUMOR)
    zn = spread(np.where((pan >= MIN_PANC_PX) & (tum == 0))[0], N_NORMAL)
    if not zt and not zn: return 0
    vol = np.asarray(nib.load(str(img_path)).dataobj)
    for z, y in [(z, "tumor") for z in zt] + [(z, "normal") for z in zn]:
        im, mk, rk = orient(window(vol[:, :, z]), lab[:, :, z] >= 1); box = crop_box(mk)
        f = OUT / y / f"msd_{case}_z{z:03d}.png"; save_crop(im, box, f)
        ROWS.append({"path": f"{y}/{f.name}", "case": case, "label": y, "source": "msd", "z": int(z),
                     "tumor_px": int(tum[z]), "pancreas_px": int(pan[z]), "crop_y0": box[0], "crop_x0": box[1], "crop_side": box[2], "rot90": rk})
    return len(zt) + len(zn)

def stream(url):
    pending, n_cases = {}, 0
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status(); r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
            for m in tf:
                name = m.name.replace("\\", "/"); base = name.split("/")[-1]
                if not m.isfile() or base.startswith("._") or not base.endswith(".nii.gz"): continue
                kind = "img" if "/imagesTr/" in name else ("lab" if "/labelsTr/" in name else None)
                if kind is None: continue
                case = base.replace(".nii.gz", ""); p = TMP / f"{case}_{kind}.nii.gz"
                with tf.extractfile(m) as src, open(p, "wb") as dst: shutil.copyfileobj(src, dst, 1 << 20)
                pending.setdefault(case, {})[kind] = p
                if len(pending[case]) == 2:
                    try:
                        process_msd(case, pending[case]["img"], pending[case]["lab"]); n_cases += 1
                        if n_cases % 20 == 0: print(f"   MSD {n_cases} cases · {len(ROWS)} crops · {time.time()-T0:.0f}s", flush=True)
                    except Exception as e:
                        print(f"   ⚠️ {case}: {type(e).__name__}: {e}")
                    for q in pending.pop(case).values(): q.unlink(missing_ok=True)
    return n_cases

done = False
for url in URLS:
    try:
        print("streaming", url, flush=True); n = stream(url); print(f"✅ MSD: {n} labelled cases"); done = n > 0
        if done: break
    except Exception as e:
        print(f"   ⚠️ {url}: {type(e).__name__}: {e} — trying next source", flush=True)
        ROWS.clear()
        for d in ("tumor", "normal"):
            for f in (OUT / d).glob("msd_*.png"): f.unlink()
assert done, "could not download MSD Task07 from any source"
shutil.rmtree(TMP, ignore_errors=True)

# ---------------- NIH Pancreas-CT (attached .npy volumes + labels) ----------------
def find_nih():
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):      # shallowest match = dataset root
        for r in sorted(glob.glob(pat)):
            if Path(r).name.lower().startswith("pancreasct") and Path(r).is_dir(): return Path(r)
    return None

nih = find_nih(); n_nih = 0
if nih is None:
    print("⚠️ tahsin/pancreasct-dataset not attached → no NIH crops (MSD crops only)")
else:
    vols = sorted(p for p in nih.rglob("*.npy") if p.parent.name.lower() == "images")
    labs = {p.stem: p for p in nih.rglob("*.npy") if p.parent.name.lower() == "labels"}
    print(f"NIH: {len(vols)} volumes, {len(labs)} label volumes", flush=True)
    for v in vols:
        if v.stem not in labs: continue
        try:
            a = np.load(v, mmap_mode="r"); l = np.load(labs[v.stem], mmap_mode="r")
            if a.ndim != 3 or l.shape != a.shape: continue
            sh = a.shape; z_ax = int(np.argmin(sh)) if len(set(sh)) == 3 else [i for i in range(3) if sh.count(sh[i]) == 1][0]
            area = np.asarray(l > 0).sum(axis=tuple(i for i in range(3) if i != z_ax))
            zs = [int(k) for k in np.argsort(-area)[:N_NIH] if area[k] >= MIN_PANC_PX]
            for k in zs:
                s = np.take(a, k, axis=z_ax).astype(np.float32); m = np.take(l, k, axis=z_ax) > 0
                img01 = window(s) if s.min() < -500 else (lambda lo, hi: np.clip((s - lo) / (hi - lo + 1e-6), 0, 1))(*np.percentile(s, [1, 99]))
                im, mk, rk = orient(img01, m); box = crop_box(mk); f = OUT / "normal" / f"nih_{v.stem}_z{k}.png"; save_crop(im, box, f)
                ROWS.append({"path": f"normal/{f.name}", "case": f"nih_{v.stem}", "label": "normal", "source": "nih", "z": k,
                             "tumor_px": 0, "pancreas_px": int(area[k]), "crop_y0": box[0], "crop_x0": box[1], "crop_side": box[2], "rot90": rk})
            n_nih += 1
            if n_nih % 20 == 0: print(f"   NIH {n_nih} volumes · {time.time()-T0:.0f}s", flush=True)
        except Exception as e:
            print(f"   ⚠️ NIH {v.stem}: {type(e).__name__}: {e}")

meta = pd.DataFrame(ROWS); meta.to_csv(OUT / "metadata.csv", index=False)
print(f"rotation used (0–3 × 90°) by source:\n{meta.groupby(['source','rot90']).size().to_string()}")
print(f"\ncrops: {len(meta)}  by source/label:\n{meta.groupby(['source','label']).size().to_string()}\n"
      f"cases: {meta.case.nunique()}  · median crop side {int(meta.crop_side.median())} px  [{(time.time()-T0)/60:.1f} min]")

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
rng = np.random.default_rng(0); fig, ax = plt.subplots(3, 8, figsize=(18, 7))
for r_, (src, lab) in enumerate([("msd", "tumor"), ("msd", "normal"), ("nih", "normal")]):
    sub = meta[(meta.source == src) & (meta.label == lab)]
    for c_ in range(8):
        ax[r_, c_].axis("off")
        if c_ < len(sub):
            row = sub.iloc[rng.integers(len(sub))]; ax[r_, c_].imshow(Image.open(OUT / row.path), cmap="gray")
            ax[r_, c_].set_title(f"{src} {lab}", fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "samples.png", dpi=80); plt.close()
(OUT / "README.txt").write_text("Pancreas-centred CT crops. MSD Task07 (label 2 = tumour) + NIH Pancreas-CT (healthy).\n"
    f"v2: fixed square crop = {CROP_FRAC} x slice width, spine-down orientation, window [{HU_LO},{HU_HI}] HU. Group by 'case' when splitting.\n")
print("✅ output:", OUT, "→ attach this notebook's output to E2P")
